# pandas: `DataFrame` concatenation

Combining `DataFrame` objects is a common operation that is well-supported by pandas. In this lesson I'll demonstrate how to leverage the [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) function to perform concatenation operations along a specified axis.

In [1]:
import numpy as np
import pandas as pd

## Setup

First, I'll instantiate a random generator by calling [`np.random.default_rng()`](https://numpy.org/doc/stable/reference/random/generator.html) function and providing a seed value. This ensures that `DataFrame` objects composed of randomly selected integer values are reproducible.

In [2]:
rng = np.random.default_rng(4)  # Best practice

# np.random.seed(4)  # seeds global NumPy RNG (legacy; no longer recommended)

Next I'll employ a list comprehension to create a list comprising three "toy" `DataFrame` instances. Each `DataFrame` is based on a NumPy one-dimensional (1D) `1x5` array comprising five randomly selected integers ranging from `1` to `70`. The columns are named `A` to `E`.

In [3]:
columns = ['A', 'B', 'C', 'D', 'E']
toys = [pd.DataFrame(rng.integers(0, 70, (1, 5)), columns=columns) for _ in range(3)]
toys

[    A   B   C   D   E
 0  50  66  61  35  65,
     A   B  C   D   E
 0  68  67  5  31  42,
     A   B   C   D   E
 0  19  26  43  56  40]

## `DataFrame` concatenation operations

Let's combine the `DataFrame` elements in `toys` by calling the [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) function.

`pandas.concat(objs, *, axis=0, join='outer', ignore_index=False, keys=None, levels=None, names=None, verify_integrity=False, sort=False, copy=None)`

This pandas function is designed to concatenate a sequence or mapping of `Series` and/or `DataFrame` objects along a specified axis (i.e., `axis=0 or axis="index"` (default), `axis=1 or axis="columns"`). The default "outer" `join` enforces a union of all axis values.

Only the `objs` parameter (the sequence or mapping to combine) is required; all other parameters are provisioned with default values and are considered optional.

### Concatenate along the index

First, I'll leverage the default behavior of `pd.concat()` and concatenate along the index of the three `DataFrame` elements in `toys`. All I need to do is pass the list to `pd.concat()` as the lone argument. The `DataFrame` returned by the function call is named `nums`.

In [4]:
nums = pd.concat(toys)  # axis=0, axis="index" (default)
nums

,A,B,C,D,E
0,50,66,61,35,65
0,68,67,5,31,42
0,19,26,43,56,40


Note the overlapping index values. If the values are not meaningful with respect to the new `DataFrame`, discard them in favor of a new index labeled `0, ..., n - 1` by passing the keyword argument `ignore_index=True` to `pd.concat()`.

Let's call the function again and instruct it to ignore the existing `DataFrame` indexes. The return value is a new `DataFrame` with no overlapping index values.

In [5]:
nums = pd.concat(toys, ignore_index=True)  # axis=0, axis="index" (default)
nums

,A,B,C,D,E
0,50,66,61,35,65
1,68,67,5,31,42
2,19,26,43,56,40


### Concatenate along the columns

You can also combine the columns of the `DataFrame` elements in `toys` and align them side by side. Instruct `pd.concat()` to concatenate along the columns axis by passing the keyword argument `axis=1` or `axis=columns`.

In [6]:
nums = pd.concat(toys, axis=1)  # or axis="columns"
nums

,A,B,C,D,E,A,B,C,D,E,A,B,C,D,E
0,50,66,61,35,65,68,67,5,31,42,19,26,43,56,40


## Back to the material world

Imagine that you have been provided with four CSV files of lottery data. Each file contains rows of winning numbers selected twice-weekly for a US nationwide lottery named _Mega Millions_. The files cover the following years:

| Interval | Filename | Location | Columns | Notes |
| :------- | :------- | :------- | :------ | :---- |
| 2023-2024 | `mega_millions-2023_24.csv` | `./data` | Draw Date, Winning Numbers, Mega Ball, Multiplier | Year 2024 limited to first 4.5 months |
| 2016-2022 | `mega_millions-2016_22.csv` | `./data` | Draw Date, Winning Numbers, Mega Ball, Multiplier | |
| 2009-2015 | `mega_millions-2009_15.csv` | `./data` | Draw Date, Winning Numbers, Mega Ball, Multiplier | |
| 2002-2008 | `mega_millions-2002_08.csv` | `./data` | Draw Date, Winning Numbers, Mega Ball, Multiplier | |

The immediate task is to combine the data sets for future analysis.

Let's create a 4-item tuple named `intervals` that references the unique segment of each filename (e.g., "2023_24"). I can then employ a list comprehension, a formatted string literal, and repeated calls to `pd.read_csv()` to read the data into a list comprising four (unnamed) `DataFrame` elements.

In [7]:
intervals = ("2023_24", "2016_22", "2009_15", "2002_08")
dataframes = [pd.read_csv(f"./data/mega_millions-{interval}.csv") for interval in intervals]
len(dataframes)  # quick check

4

## Inspect the data

Next, let's inspect each `DataFrame` in `dataframes` by calling each object's `info()` method. Doing so will provide us with a summary of the shape, column labels, non-Null counts, and dtypes of each `DataFrame`.

I'll employ a `for` loop, the `print()` function, and a formatted string literal to limit the four calls to a single cell.

❗Note that in two instances the "Multipler" column's dtype is `float64` rather than `int64`. This is likely due to the presence of `NaN` values in the column.

In [8]:
for frame in dataframes:
    print(f"{frame.info()}\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Draw Date        144 non-null    object
 1   Winning Numbers  144 non-null    object
 2   Mega Ball        144 non-null    int64 
 3   Multiplier       144 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 4.6+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Draw Date        731 non-null    object
 1   Winning Numbers  731 non-null    object
 2   Mega Ball        731 non-null    int64 
 3   Multiplier       731 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 23.0+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 728 entries, 0 to 727
Data columns (total 4 columns):
 #   Column           Non-Null Coun

I'll also call the `DataFrame.head()` and `DataFrame.tail()` methods on the first and last `DataFrame` elements to have a look at individual row data.

In [9]:
dataframes[0].head(3)  # first 3 rows

,Draw Date,Winning Numbers,Mega Ball,Multiplier
0,05/17/2024,08 17 40 60 70,3,2
1,05/14/2024,13 19 43 62 64,6,3
2,05/10/2024,13 22 26 32 65,18,4


In [10]:
dataframes[-1].tail(3)  # last 3 rows

,Draw Date,Winning Numbers,Mega Ball,Multiplier
688,05/24/2002,02 04 32 44 52,36,NaN
689,05/21/2002,04 28 39 41 44,9,NaN
690,05/17/2002,15 18 25 33 47,30,NaN


## Combine the data

Our cursory inspection of the `DataFrame` elements in the `dataframes` list suggests that each `DataFrame` shares the same structure and column labels.

Given this, let's combine the `DataFrame` elements as ordered in the list (i.e., by time interval in descending order). I'll call `pd.concat()`, pass the `dataframes` list to it, replace the current indices, and assign the return value to a variable named `data`.

If all goes well, the function call will return a new `DataFrame` that is concatenated along the index (`axis=0`) of the four `DataFrame` objects.

In [11]:
data = pd.concat(dataframes, ignore_index=True)
data

,Draw Date,Winning Numbers,Mega Ball,Multiplier
0,05/17/2024,08 17 40 60 70,3,2.0
1,05/14/2024,13 19 43 62 64,6,3.0
2,05/10/2024,13 22 26 32 65,18,4.0
3,05/07/2024,26 28 36 63 66,15,3.0
4,05/03/2024,06 13 15 53 56,11,2.0
...,...,...,...,...
2289,05/31/2002,12 28 45 46 52,47,NaN
2290,05/28/2002,06 21 22 29 32,24,NaN
2291,05/24/2002,02 04 32 44 52,36,NaN
2292,05/21/2002,04 28 39 41 44,9,NaN


## Rename column labels

An optional step, but let me show you how to replace existing column names&mdash; "Draw Date", "Winning Numbers", "Mega Ball", and "Multiplier"&mdash; by calling the [`DataFrame.rename()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html) method.

`DataFrame.rename(mapper=None, *, index=None, columns=None, axis=None, copy=None, inplace=False, level=None, errors='ignore')`

The `DataFrame.rename()` method allows you to rename either column or index labels. To target labels for replacement create a "mapper" object (e.g., a `dict` or function).

My "mapper" is a dictionary literal named `mapper` that maps new column labels to keys named after the current column labels. I'm opting for lowercase labels featuring underscores rather than spaces.

In [12]:
mapper = {
    "Draw Date": "draw_date",
    "Winning Numbers": "winning_numbers",
    "Mega Ball": "mega_ball",
    "Multiplier": "multiplier",
}

I can then pass my mapper to `DataFrame.rename()` by binding it either to the "columns" parameter or the "mapper" parameter. If I choose the latter binding I'll also need to specify the axis (e.g., `axis="columns"` or `axis=1`).

Calling the method returns a new `DataFrame` by default but I can also mutate the `DataFrame` _in place_ by passing the keyword argument `inplace=True`.

In [13]:
data.rename(columns=mapper, inplace=True)

# data.rename(mapper=mapper, axis="columns", inplace=True)  # Alternative

data.columns

Index(['draw_date', 'winning_numbers', 'mega_ball', 'multiplier'], dtype='object')

You can also access the `DataFrame.columns` property directly and assign new labels but I recommmend that you favor method assignment over property assignment whenever possible.

In [ ]:
# data.columns = ["draw_date", "winning_numbers", "mega_ball", "multiplier"]

## Write to file

Let's conclude our discussion of `DataFrame` concatenation by writing `data` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [14]:
filepath = "./data/mega_millions_combined-2002_24.csv"
data.to_csv(filepath, index=False)